## Grid Search for Analyzer Settings

Re-runs PubTrends clustering with different weights for the following measures:
 * bibliographic coupling (BC)
 * co-citations (CC)
 * direct citations (DC)
 * text citations (TC)

In [ ]:
import json
import itertools
import logging
import os

from sklearn.metrics.cluster import adjusted_mutual_info_score, v_measure_score

from pysrc.papers.pubtrends_config import AnalyzerSettings

from utils.analysis import rebuild_similarity_graph, get_direct_references_subgraph, \
                           recalculate_topic_analysis, align_clusterings_for_sklearn
from utils.io import reload_exported_analyzer, load_analyzer, load_clustering, get_review_pmids
from utils.metrics import pd_score, reg_v_score
from utils.preprocessing import preprocess_clustering, get_clustering_level

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [ ]:
OUTPUT_FILENAME = 'grid_search_min_cocitation=2_2021-03-10_01_37.json'   # Full path is optional

In [ ]:
def settings_of_interest(analyzer_settings):
    return (
        analyzer_settings.SIMILARITY_COCITATION,
        analyzer_settings.SIMILARITY_BIBLIOGRAPHIC_COUPLING,
        analyzer_settings.SIMILARITY_CITATION,
        analyzer_settings.SIMILARITY_TEXT_CITATION,
    )

In [ ]:
def run_grid_search(analyzer, subgraph, ground_truth, 
                    metrics, target_metric,
                    baseline_params, param_names, param_range):
    """
    In order to reduce number of options:
        1. Fix one of the parameters as 1, change others. 
        2. Remove fixed parameter by setting to 0.
        3. Go to step 1 for remaining subset.
    
    Example for (cocitation, bibcoupling, citation, text_citation), ch = changed in param_range: 
    
        (1, ch, ch, ch)
        (0,  1, ch, ch)
        (0,  0,  1, ch)
        (0,  0,  0,  1)
    """
    # Accumulate grid results for all hierarchy levels
    grid_results = {k: [] for k in ground_truth.keys()}
    best_results = {k: (0, None) for k in ground_truth.keys()}
    
    # Iterate the parameter to be fixed
    for i, param in enumerate(param_names):
        const_params = baseline_params.copy()
        param_grid = {}
        
        # Setup ranges for all parameters
        for j, other_param in enumerate(param_names):
            if j <= i:
                # Add parameter weight to constants as 0 or 1
                const_params[other_param] = int(j == i)
            else:
                # Add parameter to grid
                param_grid[other_param] = param_range
    
        # Iterate over obtained param grid
        changing_param_names = param_grid.keys()
        for param_values in itertools.product(*param_grid.values()):
            # Unfold grid parameters and add constant ones
            params = {k: v for k, v in zip(changing_param_names, param_values)}
            for k, v in const_params.items():
                params[k] = v

            # Apply settings to the analyzer and recalculate the partition
            settings = AnalyzerSettings(**params)
            soi = settings_of_interest(settings)
            partition = recalculate_topic_analysis(analyzer, 
                                                   graph=subgraph,
                                                   settings=settings)
            
            # Iterate over hierarchy levels to avoid re-calculating same clustering for different GTs
            for level, ground_truth_partition in ground_truth.items():
                labels_true, labels_pred = align_clusterings_for_sklearn(partition, ground_truth_partition)

                # Save partition & iterate over different metrics
                result = {'partition': partition}
                for metric in metrics:
                    result[metric.__name__] = metric(labels_true, labels_pred)

                grid_results[level].append({
                    'soi': list(soi),
                    'results': result
                })

                new_best = False
                best_score = best_results[level][0]
                if result[target_metric] > best_score:
                    best_results[level] = (result[target_metric], soi)
                    new_best = True

            print('.', end='')
    print('\n')
    
    return grid_results, best_results

In [ ]:
param_range = [0, 0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8, 16]
# param_range = [0, 1, 2]  # For quick testing

# Order matters when choosing the parameter to be fixed.
# Current order reflects intuition about usefulness of the parameters
param_names = ['SIMILARITY_COCITATION',
               'SIMILARITY_BIBLIOGRAPHIC_COUPLING',
               'SIMILARITY_CITATION',
               'SIMILARITY_TEXT_CITATION']

In [ ]:
# Turn off all post-processing of clustering
baseline_params = dict(
    TOPIC_MIN_SIZE=0,
    TOPICS_MAX_NUMBER=500
)

In [ ]:
metrics = [adjusted_mutual_info_score, v_measure_score, pd_score, reg_v_score]
target_metric = pd_score.__name__

In [ ]:
results = []

for pmid in get_review_pmids():
    clustering = load_clustering(pmid)
    analyzer = load_analyzer(pmid)
    logging.info(f'{pmid} - loaded clustering and analyzer')
    rebuild_similarity_graph(analyzer, min_cocitation=2)
    logging.info(f'{pmid} - rebuilt similarity graph with scaling')
    
    # Pre-calculate all hierarchy levels before grid search to avoid re-calculation of clusterings
    ground_truth = {}
    for level in range(1, get_clustering_level(clustering)):
        ground_truth[level] = preprocess_clustering(clustering, level, 
                                                    include_box_sections=False,
                                                    uniqueness_method='unique_only')
    logging.info(f'{pmid} - calculated ground truth for all hierarchy levels')
    
    subgraph = get_direct_references_subgraph(analyzer, pmid)
    logging.info(f'{pmid} - running grid search')
    
    grid_results, best_results = run_grid_search(analyzer, subgraph, ground_truth, 
                                                 metrics, target_metric,
                                                 baseline_params, param_names, param_range)
    
    for level, (score, soi) in best_results.items():
        logging.info(f'{pmid} - level {level} - best score: {score}, best params: {soi}')
    
    results.append({
        'pmid': pmid,
        'results': grid_results
    })
    
    logging.info(f'{pmid} - done')

In [ ]:
with open(OUTPUT_FILENAME, 'w') as f:
    json.dump(results, f)